# Credit Risk I: Hazard Rates, Survival, and Credit Spreads

**Purpose:** Validate the reduced-form default model (hazard rate framework) from MATH 5320 formula-sheet §8. All computations use deterministic synthetic fixtures. Covers constant hazard, piecewise-constant hazard, risky ZCB pricing, and credit spreads.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.credit.hazard import (
    survival, default_density, cumulative_default_prob, interval_default_prob,
    survival_piecewise, hazard_at_piecewise, cumhazard_piecewise, density_piecewise,
    risky_zcb_price, credit_spread,
)
print('Imports OK')

## Section 1 — Constant Hazard Model

Under constant hazard $\lambda$:
- Survival: $s(t) = e^{-\lambda t}$
- Default density: $p(t) = \lambda e^{-\lambda t}$
- Cumulative PD: $F(t) = 1 - e^{-\lambda t}$
- Interval PD: $P(t_1 < \tau \leq t_2) = e^{-\lambda t_1} - e^{-\lambda t_2}$

In [ ]:
lam = 0.0074
t_values = [1, 2, 3, 4, 5]

print(f'Constant hazard lambda = {lam}')
print()
print(f'{'t':>5} {'s(t)':>10} {'p(t)':>10} {'F(t)':>10} {'P(t-1<τ≤t)':>15}')
print('-' * 54)
for t in t_values:
    s   = survival(t, lam)
    p   = default_density(t, lam)
    F   = cumulative_default_prob(t, lam)
    idp = interval_default_prob(t - 1, t, lam)
    print(f'{t:>5} {s:>10.6f} {p:>10.6f} {F:>10.6f} {idp:>15.6f}')

print()
print(f'Landmark checks:')
print(f'  survival(5, 0.0074) = {survival(5, 0.0074):.6f}  (expected 0.963600)')
print(f'  P(τ≤5)             = {cumulative_default_prob(5, 0.0074):.6f}  (expected 0.036400)')
print(f'  P(3<τ≤4)           = {interval_default_prob(3, 4, 0.0074):.6f}  (expected 0.007212)')

## Section 2 — Piecewise-Constant Hazard Model

In [ ]:
grid    = [0.0, 1.0, 2.0, 10.0]
hazards = [0.010, 0.011, 0.012]

query_times = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0]

rows = []
for t in query_times:
    lam_t  = hazard_at_piecewise(t, grid, hazards)
    Lam_t  = cumhazard_piecewise(t, grid, hazards)
    s_t    = survival_piecewise(t, grid, hazards)
    p_t    = density_piecewise(t, grid, hazards)
    rows.append({'t': t, 'lambda(t)': lam_t, 'Lambda(t)': Lam_t,
                 's(t)': s_t, 'p(t)': p_t})

df_pw = pd.DataFrame(rows).set_index('t')
pd.options.display.float_format = '{:.6f}'.format
print('Piecewise hazard model — summary table:')
print(df_pw.to_string())

## Section 3 — Risky Zero-Coupon Bond and Credit Spread

Risky ZCB price (face 1):
$$V(T) = e^{-rT}[1 - \text{LGD}(1 - s(T))]$$

Credit spread:
$$S(T) = -\frac{1}{T}\ln(1 - \text{LGD}(1 - s(T)))$$

In [ ]:
r_rf = 0.055
LGD  = 0.70

print(f'r = {r_rf}, LGD = {LGD}')
print()
print(f'{'T':>5} {'s(T)':>10} {'ZCB Price':>12} {'Spread (bps)':>14}')
print('-' * 45)
for T in [1, 2, 3, 5, 10]:
    s_T   = survival_piecewise(float(T), grid, hazards)
    price = risky_zcb_price(r_rf, float(T), LGD, s_T)
    sprd  = credit_spread(float(T), LGD, s_T)
    print(f'{T:>5} {s_T:>10.6f} {price:>12.6f} {sprd*10000:>14.2f}')

## Section 4 — Survival Curve Plot

In [ ]:
t_grid_plot = np.linspace(0, 10, 300)

# Constant hazard (lambda = 0.0074)
s_const = [survival(t, 0.0074) for t in t_grid_plot]
p_const = [default_density(t, 0.0074) for t in t_grid_plot]

# Piecewise hazard
s_pw = [survival_piecewise(t, grid, hazards) for t in t_grid_plot]
p_pw = [density_piecewise(t, grid, hazards)  for t in t_grid_plot]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(t_grid_plot, s_const, label='Constant λ=0.0074', color='steelblue')
ax1.plot(t_grid_plot, s_pw,   label='Piecewise λ', color='firebrick', linestyle='--')
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Survival Probability s(t)')
ax1.set_title('Survival Curve: Constant vs Piecewise Hazard')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(t_grid_plot, p_const, label='Constant λ=0.0074', color='steelblue')
ax2.plot(t_grid_plot, p_pw,   label='Piecewise λ', color='firebrick', linestyle='--')
ax2.set_xlabel('Time (years)')
ax2.set_ylabel('Default Density p(t)')
ax2.set_title('Default Density: Constant vs Piecewise Hazard')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 5 — Validation Summary

| Function | Input | Expected | Notes |
|----------|-------|----------|-------|
| `survival(5, 0.0074)` | λ=0.0074, t=5 | 0.96368 | Course fixture |
| `cumulative_default_prob(5, 0.0074)` | | 0.03632 | 1 − survival |
| `interval_default_prob(3, 4, 0.0074)` | | 0.00721 | Between years 3 and 4 |
| `survival_piecewise(2.0, ...)` | piecewise | exp(-0.021) | Integral up to t=2 |
| `credit_spread(5, LGD=0.70, ...)` | piecewise | see table above | Basis points |

**Key insight:** Under constant hazard, the survival curve is an exponential and the default density is an exponential tilt. Piecewise hazard allows calibration to market term-structure of default probabilities (or CDS spreads at multiple tenors).